# 06 — CUI TF-IDF Similarity

Goal:

1. Use existing `cui_set` files generated from `Title + Summary`.
2. Weight CUIs by TF-IDF so rare CUIs matter more.
3. Find top similar pairs:
   - human × mouse
   - within human
   - within mouse
4. Add metadata/source flags.
5. Remove exact duplicate CUI/summary pairs.
6. Compare CUI similarity with raw text similarity.

In [1]:
from pathlib import Path
import ast
import heapq
import re
import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 1. Configuration

In [2]:
# Input files
HUMAN_CUI_PATH = Path("metadata/human_with_cuis.pkl")
MOUSE_CUI_PATH = Path("metadata/mouse_with_cuis.pkl")

# Optional source metadata files generated by src/find_pmid.py
HUMAN_SOURCE_PATH = Path("metadata/gse_pmid_subseries_of_human.tsv")
MOUSE_SOURCE_PATH = Path("metadata/gse_pmid_subseries_of_mouse.tsv")

# Output directory
RESULT_DIR = Path("similarity-results")
RESULT_DIR.mkdir(exist_ok=True)

# Similarity settings
TOP_N = 1000
CHUNK_SIZE = 100

# Output files
HUMAN_MOUSE_OUT = RESULT_DIR / "top_1000_human_mouse_cui_tfidf_pairs.csv"
WITHIN_HUMAN_OUT = RESULT_DIR / "top_1000_within_human_cui_tfidf_pairs.csv"
WITHIN_MOUSE_OUT = RESULT_DIR / "top_1000_within_mouse_cui_tfidf_pairs.csv"

ALL_PAIRS_OUT = RESULT_DIR / "top_all_cui_tfidf_similarity_pairs.csv"
WITH_FLAGS_OUT = RESULT_DIR / "top_all_cui_tfidf_pairs_with_metadata_source_flags.csv"
EXCLUDED_EXACT_OUT = RESULT_DIR / "excluded_exact_cui_or_summary_pairs.csv"
FILTERED_EXACT_OUT = RESULT_DIR / "top_all_cui_tfidf_pairs_filtered_exact.csv"
TEXT_SIM_OUT = RESULT_DIR / "top_all_cui_tfidf_pairs_with_text_similarity.csv"

## 2. Helper functions

In [3]:
def ensure_cui_set(x):
    """Return a Python set for CUI data stored as set/list/string."""
    if isinstance(x, set):
        return x
    if isinstance(x, list):
        return set(x)
    if isinstance(x, str):
        try:
            return set(ast.literal_eval(x))
        except Exception:
            return set()
    return set()


def normalize_text(x, remove_punctuation=False):
    """Normalize text for exact/near-exact comparison."""
    if pd.isna(x):
        return ""
    x = str(x).lower()
    x = re.sub(r"\s+", " ", x)
    if remove_punctuation:
        x = re.sub(r"[^\w\s]", "", x)
    return x.strip()


def split_source_field(x):
    """Split source metadata fields such as PMID/SRA/BioProject into comparable sets."""
    if pd.isna(x):
        return set()
    if isinstance(x, list):
        return set(map(str, x))
    return set(i.strip() for i in str(x).split(";") if i.strip())


def shared_cuis(cui_set_1, cui_set_2):
    return sorted(cui_set_1 & cui_set_2)


def shared_idf_sum(cuis, idf_lookup):
    return sum(idf_lookup.get(cui, 0) for cui in cuis)

## 3. Load existing CUI sets

In [4]:
human_df = pd.read_pickle(HUMAN_CUI_PATH)
mouse_df = pd.read_pickle(MOUSE_CUI_PATH)

human_df["cui_set"] = human_df["cui_set"].apply(ensure_cui_set)
mouse_df["cui_set"] = mouse_df["cui_set"].apply(ensure_cui_set)

print("Human GSEs:", len(human_df))
print("Mouse GSEs:", len(mouse_df))
print("Human columns:", list(human_df.columns))
print("Mouse columns:", list(mouse_df.columns))

Human GSEs: 3395
Mouse GSEs: 4066
Human columns: ['gse_id', 'species', 'title', 'summary', 'text', 'cui_set']
Mouse columns: ['gse_id', 'species', 'title', 'summary', 'text', 'cui_set']


## 4. Build CUI TF-IDF matrices

In [5]:
human_df["cui_doc"] = human_df["cui_set"].apply(lambda s: " ".join(sorted(s)))
mouse_df["cui_doc"] = mouse_df["cui_set"].apply(lambda s: " ".join(sorted(s)))

all_cui_docs = pd.concat(
    [human_df["cui_doc"], mouse_df["cui_doc"]],
    ignore_index=True
)

cui_vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    binary=True,
    use_idf=True,
    smooth_idf=True,
    norm="l2"
)

X_all = cui_vectorizer.fit_transform(all_cui_docs)

n_human = len(human_df)
X_human = X_all[:n_human]
X_mouse = X_all[n_human:]

cui_idf = dict(zip(cui_vectorizer.get_feature_names_out(), cui_vectorizer.idf_))

print("Human TF-IDF matrix:", X_human.shape)
print("Mouse TF-IDF matrix:", X_mouse.shape)
print("Unique CUIs:", len(cui_idf))

Human TF-IDF matrix: (3395, 33968)
Mouse TF-IDF matrix: (4066, 33968)
Unique CUIs: 33968


## 5. Find top CUI-TF-IDF pairs

In [6]:
def compute_top_cui_tfidf_pairs(
    left_df,
    right_df,
    X_left,
    X_right,
    comparison_label,
    output_path,
    top_n=1000,
    chunk_size=100,
    within=False
):
    """
    Compute top CUI-TF-IDF cosine similarity pairs.

    For within-species comparisons, set within=True to remove:
    - self pairs
    - duplicated reverse pairs
    """

    if output_path.exists():
        result = pd.read_csv(output_path)
        print(f"Loaded existing file: {output_path}")
        return result

    heap = []
    counter = 0

    for start in tqdm(range(0, X_left.shape[0], chunk_size), desc=comparison_label):
        end = min(start + chunk_size, X_left.shape[0])

        sim_chunk = (X_left[start:end] @ X_right.T).toarray()

        if within:
            for local_i in range(sim_chunk.shape[0]):
                global_i = start + local_i
                sim_chunk[local_i, :global_i + 1] = 0

        flat = sim_chunk.ravel()
        candidate_n = min(top_n, flat.size)
        candidate_idx = np.argpartition(flat, -candidate_n)[-candidate_n:]

        for idx in candidate_idx:
            local_i, j = np.unravel_index(idx, sim_chunk.shape)
            i = start + local_i
            score = sim_chunk[local_i, j]

            if score <= 0:
                continue

            left_cuis = left_df.iloc[i]["cui_set"]
            right_cuis = right_df.iloc[j]["cui_set"]
            shared = shared_cuis(left_cuis, right_cuis)

            row = {
                "comparison": comparison_label,
                "gse_1": left_df.iloc[i]["gse_id"],
                "gse_2": right_df.iloc[j]["gse_id"],
                "tfidf_cosine": score,
                "shared_cui_count": len(shared),
                "shared_idf_sum": shared_idf_sum(shared, cui_idf),
                "shared_cuis": shared,
                "title_1": left_df.iloc[i]["title"],
                "title_2": right_df.iloc[j]["title"],
            }

            item = (score, row["shared_idf_sum"], row["shared_cui_count"], counter, row)
            counter += 1

            if len(heap) < top_n:
                heapq.heappush(heap, item)
            else:
                heapq.heappushpop(heap, item)

    result = (
        pd.DataFrame([item[-1] for item in heap])
        .sort_values(["tfidf_cosine", "shared_idf_sum", "shared_cui_count"], ascending=False)
        .reset_index(drop=True)
    )

    result.to_csv(output_path, index=False)
    print(f"Saved new file: {output_path}")

    return result

In [7]:
top_human_mouse = compute_top_cui_tfidf_pairs(
    left_df=human_df,
    right_df=mouse_df,
    X_left=X_human,
    X_right=X_mouse,
    comparison_label="human_mouse",
    output_path=HUMAN_MOUSE_OUT,
    top_n=TOP_N,
    chunk_size=CHUNK_SIZE,
    within=False
)

top_within_human = compute_top_cui_tfidf_pairs(
    left_df=human_df,
    right_df=human_df,
    X_left=X_human,
    X_right=X_human,
    comparison_label="within_human",
    output_path=WITHIN_HUMAN_OUT,
    top_n=TOP_N,
    chunk_size=CHUNK_SIZE,
    within=True
)

top_within_mouse = compute_top_cui_tfidf_pairs(
    left_df=mouse_df,
    right_df=mouse_df,
    X_left=X_mouse,
    X_right=X_mouse,
    comparison_label="within_mouse",
    output_path=WITHIN_MOUSE_OUT,
    top_n=TOP_N,
    chunk_size=CHUNK_SIZE,
    within=True
)

human_mouse:   0%|          | 0/34 [00:00<?, ?it/s]

human_mouse: 100%|██████████| 34/34 [00:04<00:00,  7.00it/s]


Saved new file: similarity-results/top_1000_human_mouse_cui_tfidf_pairs.csv


within_human: 100%|██████████| 34/34 [00:04<00:00,  7.25it/s]


Saved new file: similarity-results/top_1000_within_human_cui_tfidf_pairs.csv


within_mouse: 100%|██████████| 41/41 [00:05<00:00,  7.40it/s]

Saved new file: similarity-results/top_1000_within_mouse_cui_tfidf_pairs.csv


## 6. Combine all top pairs

In [8]:
all_pairs = pd.concat(
    [top_human_mouse, top_within_human, top_within_mouse],
    ignore_index=True
)

all_pairs = (
    all_pairs
    .sort_values(["tfidf_cosine", "shared_idf_sum", "shared_cui_count"], ascending=False)
    .drop_duplicates(subset=["comparison", "gse_1", "gse_2"])
    .reset_index(drop=True)
)

all_pairs.to_csv(ALL_PAIRS_OUT, index=False)

print("All top pairs:", len(all_pairs))
all_pairs.head(10)

All top pairs: 3000


,comparison,gse_1,gse_2,tfidf_cosine,shared_cui_count,shared_idf_sum,shared_cuis,title_1,title_2
0,within_mouse,GSE101623,GSE101624,1.0,165,965.694778,"[C0001554, C0001563, C0005884, C0014457, C0014...",The neuropeptide Neuromedin U stimulates innat...,The neuropeptide Neuromedin U stimulates innat...
1,within_mouse,GSE76864,GSE76865,1.0,109,622.012226,"[C0001688, C0003241, C0003242, C0004561, C0007...",Independent roles of switching and hypermutati...,Independent roles of switching and hypermutati...
2,within_human,GSE94528,GSE94999,1.0,109,564.937214,"[C0001272, C0001688, C0004561, C0006826, C0007...","H3B-8800, a novel oral splicing modulator, ind...","H3B-8800, a novel oral splicing modulator, ind..."
3,within_human,GSE81074,GSE81080,1.0,108,547.524853,"[C0002793, C0002874, C0004561, C0005839, C0005...",Differentiation of human embryonic stem cells ...,Differentiation of human embryonic stem cells ...
4,within_mouse,GSE73559,GSE73560,1.0,110,565.120041,"[C0004561, C0007634, C0011155, C0011377, C0011...",Gene expression analysis to identify Klf2 targ...,Gene expression analysis to identify Klf2 targ...
5,within_mouse,GSE77736,GSE77740,1.0,103,539.182528,"[C0001272, C0001779, C0001811, C0004561, C0005...",Single Novel single cell assay reveals progres...,Single Novel single cell assay reveals progres...
6,human_mouse,GSE103658,GSE103725,1.0,101,496.886869,"[C0001792, C0002976, C0015609, C0017262, C0019...",Expression changes in Melanomas pre MAPKi trea...,Expression changes in Melanomas pre MAPKi trea...
7,within_mouse,GSE103185,GSE104325,1.0,85,417.046933,"[C0001779, C0004909, C0006104, C0007613, C0012...",Early-life gene expression in neurons modulate...,Early-life gene expression in neurons modulate...
8,within_human,GSE96562,GSE96563,1.0,149,773.952612,"[C0003320, C0003334, C0003341, C0004561, C0005...",Single cell RNA-seq reveals expansion of IGRP-...,Single cell RNA-seq reveals expansion of IGRP-...
9,within_human,GSE81497,GSE81498,1.0,133,657.067069,"[C0002684, C0003015, C0005495, C0006826, C0007...",Multiple mechanisms disrupt let-7 miRNA biogen...,Multiple mechanisms disrupt let-7 miRNA biogen...


## 7. Add metadata and source flags

In [9]:
# Metadata lookup from existing human/mouse data
human_lookup = human_df.set_index("gse_id").to_dict(orient="index")
mouse_lookup = mouse_df.set_index("gse_id").to_dict(orient="index")
metadata_lookup = {**human_lookup, **mouse_lookup}


def get_metadata(gse, field):
    return metadata_lookup.get(gse, {}).get(field, np.nan)


def get_cui_set(gse):
    return ensure_cui_set(get_metadata(gse, "cui_set"))


pairs_with_flags = all_pairs.copy()

pairs_with_flags["title_1_full"] = pairs_with_flags["gse_1"].apply(lambda g: get_metadata(g, "title"))
pairs_with_flags["title_2_full"] = pairs_with_flags["gse_2"].apply(lambda g: get_metadata(g, "title"))
pairs_with_flags["summary_1"] = pairs_with_flags["gse_1"].apply(lambda g: get_metadata(g, "summary"))
pairs_with_flags["summary_2"] = pairs_with_flags["gse_2"].apply(lambda g: get_metadata(g, "summary"))

pairs_with_flags["summary_1_norm"] = pairs_with_flags["summary_1"].apply(lambda x: normalize_text(x, remove_punctuation=True))
pairs_with_flags["summary_2_norm"] = pairs_with_flags["summary_2"].apply(lambda x: normalize_text(x, remove_punctuation=True))

pairs_with_flags["same_summary"] = pairs_with_flags["summary_1_norm"] == pairs_with_flags["summary_2_norm"]
pairs_with_flags["same_cui_set"] = pairs_with_flags.apply(
    lambda r: get_cui_set(r["gse_1"]) == get_cui_set(r["gse_2"]),
    axis=1
)

print("Exact same summary pairs:", pairs_with_flags["same_summary"].sum())
print("Exact same CUI-set pairs:", pairs_with_flags["same_cui_set"].sum())

Exact same summary pairs: 452
Exact same CUI-set pairs: 156


In [10]:
# Load source metadata if available
if HUMAN_SOURCE_PATH.exists() and MOUSE_SOURCE_PATH.exists():
    human_source = pd.read_csv(HUMAN_SOURCE_PATH, sep="\t")
    mouse_source = pd.read_csv(MOUSE_SOURCE_PATH, sep="\t")

    source_df = pd.concat([human_source, mouse_source], ignore_index=True)
    source_lookup = source_df.set_index("gse").to_dict(orient="index")

    print("Loaded source metadata:")
    print("Human source rows:", len(human_source))
    print("Mouse source rows:", len(mouse_source))
else:
    source_lookup = {}
    print("Source metadata files not found.")
    print("Expected files:")
    print(HUMAN_SOURCE_PATH)
    print(MOUSE_SOURCE_PATH)


def get_source(gse, field):
    return source_lookup.get(gse, {}).get(field, np.nan)


def source_overlap(gse_1, gse_2, field):
    values_1 = split_source_field(get_source(gse_1, field))
    values_2 = split_source_field(get_source(gse_2, field))
    return sorted(values_1 & values_2)


source_fields = ["pmid", "subseries", "superseries", "affiliation", "BioProject", "SRA"]

for field in source_fields:
    pairs_with_flags[f"{field}_1"] = pairs_with_flags["gse_1"].apply(lambda g: get_source(g, field))
    pairs_with_flags[f"{field}_2"] = pairs_with_flags["gse_2"].apply(lambda g: get_source(g, field))
    pairs_with_flags[f"shared_{field}"] = pairs_with_flags.apply(
        lambda r: source_overlap(r["gse_1"], r["gse_2"], field),
        axis=1
    )
    pairs_with_flags[f"same_{field}"] = pairs_with_flags[f"shared_{field}"].apply(lambda x: len(x) > 0)


def same_source_label(row):
    if row["same_pmid"] and row["same_subseries"]:
        return pd.Series({"same_source_prediction": True, "same_source_confidence": "Very High"})
    if row["same_pmid"] or row["same_subseries"]:
        return pd.Series({"same_source_prediction": True, "same_source_confidence": "High"})
    if row["same_SRA"] or row["same_BioProject"]:
        return pd.Series({"same_source_prediction": True, "same_source_confidence": "Moderate"})
    return pd.Series({"same_source_prediction": False, "same_source_confidence": "Unknown"})


pairs_with_flags[["same_source_prediction", "same_source_confidence"]] = pairs_with_flags.apply(
    same_source_label,
    axis=1
)

pairs_with_flags["same_source_confidence"].value_counts(dropna=False)

Loaded source metadata:
Human source rows: 3395
Mouse source rows: 4064


same_source_confidence
Unknown      2237
Very High     578
High          185
Name: count, dtype: int64

## 8. Remove exact CUI-set and exact summary pairs

In [11]:
excluded_exact = pairs_with_flags[
    pairs_with_flags["same_cui_set"] | pairs_with_flags["same_summary"]
].copy()

filtered_pairs = pairs_with_flags[
    ~(pairs_with_flags["same_cui_set"] | pairs_with_flags["same_summary"])
].copy().reset_index(drop=True)

pairs_with_flags.to_csv(WITH_FLAGS_OUT, index=False)
excluded_exact.to_csv(EXCLUDED_EXACT_OUT, index=False)
filtered_pairs.to_csv(FILTERED_EXACT_OUT, index=False)

print("Pairs before filtering:", len(pairs_with_flags))
print("Excluded exact CUI-set or summary pairs:", len(excluded_exact))
print("Pairs after filtering:", len(filtered_pairs))
print("Saved:", FILTERED_EXACT_OUT)

# filtered_pairs.head(10)

Pairs before filtering: 3000
Excluded exact CUI-set or summary pairs: 470
Pairs after filtering: 2530
Saved: similarity-results/top_all_cui_tfidf_pairs_filtered_exact.csv


## 9. Compare raw text similarity with CUI-TF-IDF similarity

In [12]:
# Use a separate dataframe for text-side comparison
analysis_pairs = filtered_pairs.copy()
analysis_pairs = analysis_pairs.rename(columns={"tfidf_cosine": "cui_tfidf_cosine"})

analysis_pairs["title_1_norm"] = analysis_pairs["title_1_full"].apply(normalize_text)
analysis_pairs["title_2_norm"] = analysis_pairs["title_2_full"].apply(normalize_text)
analysis_pairs["summary_1_norm"] = analysis_pairs["summary_1"].apply(normalize_text)
analysis_pairs["summary_2_norm"] = analysis_pairs["summary_2"].apply(normalize_text)

analysis_pairs["text_1_norm"] = (
    analysis_pairs["title_1_norm"] + " " + analysis_pairs["summary_1_norm"]
).str.strip()

analysis_pairs["text_2_norm"] = (
    analysis_pairs["title_2_norm"] + " " + analysis_pairs["summary_2_norm"]
).str.strip()

In [13]:
# Edit-distance similarity: detects near-duplicate metadata
try:
    from rapidfuzz.distance import DamerauLevenshtein
except ImportError as exc:
    raise ImportError("Please install rapidfuzz first: pip install rapidfuzz") from exc


def damerau_similarity(text_1, text_2):
    text_1 = normalize_text(text_1)
    text_2 = normalize_text(text_2)

    max_len = max(len(text_1), len(text_2))
    if max_len == 0:
        return 0

    distance = DamerauLevenshtein.distance(text_1, text_2)
    return 1 - distance / max_len


analysis_pairs["title_damerau_sim"] = analysis_pairs.apply(
    lambda r: damerau_similarity(r["title_1_norm"], r["title_2_norm"]),
    axis=1
)

analysis_pairs["summary_damerau_sim"] = analysis_pairs.apply(
    lambda r: damerau_similarity(r["summary_1_norm"], r["summary_2_norm"]),
    axis=1
)

analysis_pairs["text_damerau_sim"] = analysis_pairs.apply(
    lambda r: damerau_similarity(r["text_1_norm"], r["text_2_norm"]),
    axis=1
)

In [14]:
# Text TF-IDF cosine: lexical/semantic text-side baseline
def paired_text_tfidf_cosine(texts_1, texts_2):
    all_texts = pd.concat([texts_1, texts_2], ignore_index=True)

    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        min_df=1,
        ngram_range=(1, 2)
    )

    X = vectorizer.fit_transform(all_texts)
    n = len(texts_1)

    return cosine_similarity(X[:n], X[n:]).diagonal()


analysis_pairs["title_text_tfidf_cosine"] = paired_text_tfidf_cosine(
    analysis_pairs["title_1_norm"],
    analysis_pairs["title_2_norm"]
)

analysis_pairs["summary_text_tfidf_cosine"] = paired_text_tfidf_cosine(
    analysis_pairs["summary_1_norm"],
    analysis_pairs["summary_2_norm"]
)

analysis_pairs["text_tfidf_cosine"] = paired_text_tfidf_cosine(
    analysis_pairs["text_1_norm"],
    analysis_pairs["text_2_norm"]
)

In [15]:
# Flags and disagreement classes
# CUI-TF-IDF similarity / raw text TF-IDF similarity / Damerau-Levenshtein similarity is in top 25%
analysis_pairs["near_same_title"] = analysis_pairs["title_damerau_sim"] >= 0.90
analysis_pairs["near_same_summary"] = analysis_pairs["summary_damerau_sim"] >= 0.95
analysis_pairs["near_same_text"] = analysis_pairs["text_damerau_sim"] >= 0.95

cui_high_cutoff = analysis_pairs["cui_tfidf_cosine"].quantile(0.75)
text_high_cutoff = analysis_pairs["text_tfidf_cosine"].quantile(0.75)
edit_high_cutoff = analysis_pairs["text_damerau_sim"].quantile(0.75)

analysis_pairs["cui_high"] = analysis_pairs["cui_tfidf_cosine"] >= cui_high_cutoff  # highly overlapped biomedical concepts in paired GSEs
analysis_pairs["text_tfidf_high"] = analysis_pairs["text_tfidf_cosine"] >= text_high_cutoff # text are ~ exactly the same (in spelling)
analysis_pairs["edit_high"] = analysis_pairs["text_damerau_sim"] >= edit_high_cutoff    # also show text similarity, but more on semantic side


def classify_similarity_pattern(row):
    if row["cui_high"] and row["edit_high"]:
        # same-project / near-duplicate metadata / same summary / same publication companion dataset
        return "high_cui_high_edit_near_duplicate"
    if row["cui_high"] and (not row["edit_high"]) and row["text_tfidf_high"]:
        # similar biological topic (what we are looking for?)
        return "high_cui_low_edit_high_text"
    if row["cui_high"] and (not row["edit_high"]) and (not row["text_tfidf_high"]):
        # 1. cui-tf-idf identifies bio concept similarities that cannot find via raw text
        # 2. CUI extraction / generic CUI overlap -> false positive
        # need manual check
        return "high_cui_low_text_potential_cui_specific"
    if (not row["cui_high"]) and row["text_tfidf_high"]:
        # could be helpful in identifying the weakness of cui-tf-idf approach
        return "low_cui_high_text"
    return "other"


analysis_pairs["similarity_pattern"] = analysis_pairs.apply(classify_similarity_pattern, axis=1)

analysis_pairs["similarity_pattern"].value_counts()

similarity_pattern
other                                       1695
high_cui_high_edit_near_duplicate            432
low_cui_high_text                            202
high_cui_low_text_potential_cui_specific     169
high_cui_low_edit_high_text                   32
Name: count, dtype: int64

## 10. Summary tables and outputs

In [ ]:
score_columns = [
    "cui_tfidf_cosine",
    "title_damerau_sim",
    "summary_damerau_sim",
    "text_damerau_sim",
    "title_text_tfidf_cosine",
    "summary_text_tfidf_cosine",
    "text_tfidf_cosine"
]

score_summary = analysis_pairs[score_columns].describe()
score_correlation = analysis_pairs[score_columns].corr()

display(score_summary)
display(score_correlation)

# cui_tfidf_cosine vs summary / text -> pretty postitively correlated

,cui_tfidf_cosine,title_damerau_sim,summary_damerau_sim,text_damerau_sim,title_text_tfidf_cosine,summary_text_tfidf_cosine,text_tfidf_cosine
count,2530.000000,2530.000000,2530.000000,2530.000000,2530.000000,2530.000000,2530.000000
mean,0.437250,0.347982,0.343747,0.354715,0.150495,0.183684,0.193274
std,0.146316,0.204578,0.207199,0.186965,0.218565,0.241136,0.222172
min,0.306077,0.038835,0.033967,0.082949,0.000000,0.000000,0.002192
25%,0.341341,0.217391,0.223847,0.245685,0.007218,0.026468,0.040028
50%,0.384013,0.265976,0.264608,0.282353,0.055549,0.074556,0.097226
75%,0.464376,0.400000,0.377069,0.392335,0.197421,0.235778,0.263553
max,0.999112,1.000000,0.999186,0.999164,1.000000,1.000000,0.998622


,cui_tfidf_cosine,title_damerau_sim,summary_damerau_sim,text_damerau_sim,title_text_tfidf_cosine,summary_text_tfidf_cosine,text_tfidf_cosine
cui_tfidf_cosine,1.000000,0.595830,0.764184,0.796292,0.635234,0.798547,0.817085
title_damerau_sim,0.595830,1.000000,0.564015,0.699008,0.884572,0.584787,0.674329
summary_damerau_sim,0.764184,0.564015,1.000000,0.961619,0.534944,0.897539,0.836068
text_damerau_sim,0.796292,0.699008,0.961619,1.000000,0.653793,0.889444,0.873400
title_text_tfidf_cosine,0.635234,0.884572,0.534944,0.653793,1.000000,0.611026,0.747326
summary_text_tfidf_cosine,0.798547,0.584787,0.897539,0.889444,0.611026,1.000000,0.961167
text_tfidf_cosine,0.817085,0.674329,0.836068,0.873400,0.747326,0.961167,1.000000


In [23]:
# three CUI-high groups we want to inspect
target_patterns = [
    "high_cui_high_edit_near_duplicate",
    "high_cui_low_edit_high_text",
    "high_cui_low_text_potential_cui_specific"
]

interesting_cui_pairs = (
    analysis_pairs[
        analysis_pairs["similarity_pattern"].isin(target_patterns)
    ]
    .sort_values(
        ["similarity_pattern", "cui_tfidf_cosine"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

interesting_cui_pairs["similarity_pattern"].value_counts()

similarity_pattern
high_cui_high_edit_near_duplicate           432
high_cui_low_text_potential_cui_specific    169
high_cui_low_edit_high_text                  32
Name: count, dtype: int64

In [24]:
near_duplicate_pairs = interesting_cui_pairs[
    interesting_cui_pairs["similarity_pattern"] == "high_cui_high_edit_near_duplicate"
]

text_supported_cui_pairs = interesting_cui_pairs[
    interesting_cui_pairs["similarity_pattern"] == "high_cui_low_edit_high_text"
]

cui_specific_pairs = interesting_cui_pairs[
    interesting_cui_pairs["similarity_pattern"] == "high_cui_low_text_potential_cui_specific"
]

In [27]:
text_supported_cui_pairs[text_supported_cui_pairs['comparison'] == 'human_mouse']

,comparison,gse_1,gse_2,cui_tfidf_cosine,shared_cui_count,shared_idf_sum,shared_cuis,title_1,title_2,title_1_full,...,title_text_tfidf_cosine,summary_text_tfidf_cosine,text_tfidf_cosine,near_same_title,near_same_summary,near_same_text,cui_high,text_tfidf_high,edit_high,similarity_pattern
447,human_mouse,GSE41009,GSE36799,0.544687,43,174.196536,"[C0007634, C0008111, C0012854, C0017262, C0027...",Divergent transcription of lncRNA/mRNA gene pa...,Long non-coding RNAs from divergent transcript...,Divergent transcription of lncRNA/mRNA gene pa...,...,0.181591,0.297684,0.349701,False,False,False,True,True,False,high_cui_low_edit_high_text
449,human_mouse,GSE33480,GSE37909,0.528760,64,344.658220,"[C0002684, C0004793, C0006360, C0006556, C0008...",RNA-seq from ENCODE/Caltech,RNA-seq from ENCODE/Caltech (Mouse),RNA-seq from ENCODE/Caltech,...,0.854296,0.526618,0.531555,False,False,False,True,True,False,high_cui_low_edit_high_text


In [ ]:
interesting_cui_pairs.to_csv(
    "similarity-results/interesting_cui_similarity_patterns.csv",
    index=False
)

near_duplicate_pairs.to_csv(
    "similarity-results/interesting_near_duplicate_cui_pairs.csv",
    index=False
)

text_supported_cui_pairs.to_csv(
    "similarity-results/interesting_text_supported_cui_pairs.csv",
    index=False
)

cui_specific_pairs.to_csv(
    "similarity-results/interesting_cui_specific_pairs.csv",
    index=False
)

In [22]:
# for cui pipeline diagnosis

# text_high_cui_lower = (
#     analysis_pairs[
#         (~analysis_pairs["cui_high"]) &
#         (analysis_pairs["text_tfidf_high"])
#     ]
#     .sort_values("text_tfidf_cosine", ascending=False)
# )

# text_high_cui_lower[[
#     "comparison",
#     "gse_1",
#     "gse_2",
#     "cui_tfidf_cosine",
#     "text_tfidf_cosine",
#     "title_damerau_sim",
#     "summary_damerau_sim",
#     "title_1_full",
#     "title_2_full"
# ]].head(20)

In [19]:
analysis_pairs.to_csv(TEXT_SIM_OUT, index=False)

print("Saved:", TEXT_SIM_OUT)

Saved: similarity-results/top_all_cui_tfidf_pairs_with_text_similarity.csv
